In [ ]:
import os

# Save models locally
os.makedirs('../models/demand_forecasting', exist_ok=True)

# Save best sklearn model
best_model_name_clean = comparison_df.iloc[0]['model'].lower().replace(' ', '_')
best_model_obj = locals()[f'{best_model_name_clean}_pipeline']

if best_model_name_clean == 'lstm':
    best_model_obj.save('../models/demand_forecasting/best_lstm_model.h5')
    print(f"Saved LSTM model")
else:
    joblib.dump(best_model_obj, f'../models/demand_forecasting/best_{best_model_name_clean}_model.pkl')
    print(f"Saved {comparison_df.iloc[0]['model']} model")

# Save scaler
joblib.dump(scaler, '../models/demand_forecasting/scaler.pkl')
print("Saved scaler")

# Create summary report
summary = f"""
# Demand Forecasting Model Training Summary

## Dataset
- Store Item Demand Forecasting
- Total samples: {len(df_clean)}
- Stores: {df_clean['store'].nunique()}
- Items: {df_clean['item'].nunique()}
- Date range: {df_clean['date'].min()} to {df_clean['date'].max()}
- Training samples: {len(df_train)} | Validation: {len(df_val)} | Test: {len(df_test)}

## Feature Engineering
- Base features: 4 (date, store, item, demand)
- Lag windows: 7, 14, 30 days
- Rolling windows: 7, 14, 30 days
- Temporal features: day_of_week, month, quarter, day_of_year, is_weekend
- Total engineered features: {len(feature_cols)}

## Best Model: {comparison_df.iloc[0]['model']}
- RMSE: {comparison_df.iloc[0]['rmse']:.4f}
- MAE: {comparison_df.iloc[0]['mae']:.4f}
- R² Score: {comparison_df.iloc[0]['r2']:.4f}

## Model Comparison
{comparison_df.to_string(index=False)}

## Recommendations
1. {best_model_name.split()[0]} model performs best with R² = {comparison_df.iloc[0]['r2']:.4f}
2. Residuals show mean ≈ 0, indicating unbiased predictions
3. Consider ensemble methods combining LSTM with tree-based models
4. Monitor for concept drift as new sales patterns emerge
5. Implement online learning for continuous model updates

## Next Steps
1. Deploy best model to production
2. Set up monitoring for prediction drift
3. Plan retraining schedule (quarterly or semi-annual)
4. Implement A/B testing for model updates
5. Add explainability (SHAP values) for stakeholder insights
"""

with open('../models/demand_forecasting/SUMMARY.md', 'w') as f:
    f.write(summary)

print("\nSummary Report Generated!")
print(summary)

## Step 15: Model Persistence & Summary

In [ ]:
# Analyze residuals for best model (XGBoost)
best_model_name = comparison_df.iloc[0]['model']
best_predictions = locals()[f'y_pred_{best_model_name.lower().replace(" ", "_")}']
residuals = y_test - best_predictions

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Residuals distribution
axes[0, 0].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_title('Residuals Distribution')
axes[0, 0].set_xlabel('Residual')
axes[0, 0].set_ylabel('Frequency')

# Q-Q plot (normal probability plot)
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot')

# Residuals vs Fitted
axes[1, 0].scatter(best_predictions, residuals, alpha=0.5)
axes[1, 0].axhline(y=0, color='r', linestyle='--')
axes[1, 0].set_title('Residuals vs Fitted Values')
axes[1, 0].set_xlabel('Fitted Values')
axes[1, 0].set_ylabel('Residuals')
axes[1, 0].grid(True, alpha=0.3)

# Residuals over time
axes[1, 1].plot(residuals, marker='o', alpha=0.5)
axes[1, 1].axhline(y=0, color='r', linestyle='--')
axes[1, 1].set_title('Residuals Over Time')
axes[1, 1].set_xlabel('Sample')
axes[1, 1].set_ylabel('Residual')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nResiduals Summary for {best_model_name}:")
print(f"  Mean: {residuals.mean():.4f}")
print(f"  Std: {residuals.std():.4f}")
print(f"  Min: {residuals.min():.4f}")
print(f"  Max: {residuals.max():.4f}")

## Step 14: Residuals Analysis

In [ ]:
# Configure MLflow
mlflow.set_experiment("Demand-Forecasting-Store-Item")

# Log each model to MLflow
models_to_log = {
    'Linear Regression': (lr_pipeline, y_pred_lr, y_test, r2_lr),
    'Random Forest': (rf_pipeline, y_pred_rf, y_test, r2_rf),
    'XGBoost': (xgb_pipeline, y_pred_xgb, y_test, r2_xgb),
    'LSTM': (lstm_model, y_pred_lstm, y_test_lstm, r2_lstm)
}

for model_name, (model, predictions, targets, r2) in models_to_log.items():
    with mlflow.start_run(run_name=model_name):
        rmse = np.sqrt(mean_squared_error(targets, predictions))
        mae = mean_absolute_error(targets, predictions)
        
        # Log metrics
        mlflow.log_metric('rmse', rmse)
        mlflow.log_metric('mae', mae)
        mlflow.log_metric('r2', r2)
        
        # Log params
        mlflow.log_param('dataset', 'Store Item Demand')
        mlflow.log_param('test_size', 0.15)
        mlflow.log_param('random_state', 42)
        
        # Log model
        if isinstance(model, Sequential):
            mlflow.keras.log_model(model, 'model')
        else:
            mlflow.sklearn.log_model(model, 'model')
        
        print(f"Logged {model_name} to MLflow")

print("\nAll models logged to MLflow!")

## Step 13: MLflow Integration

In [ ]:
# Plot predictions vs actual
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Linear Regression
axes[0, 0].plot(y_test, label='Actual', alpha=0.7, linewidth=2)
axes[0, 0].plot(y_pred_lr, label='Predicted', alpha=0.7, linewidth=2)
axes[0, 0].set_title(f'Linear Regression (R²={r2_lr:.4f})')
axes[0, 0].set_xlabel('Sample')
axes[0, 0].set_ylabel('Demand')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Random Forest
axes[0, 1].plot(y_test, label='Actual', alpha=0.7, linewidth=2)
axes[0, 1].plot(y_pred_rf, label='Predicted', alpha=0.7, linewidth=2)
axes[0, 1].set_title(f'Random Forest (R²={r2_rf:.4f})')
axes[0, 1].set_xlabel('Sample')
axes[0, 1].set_ylabel('Demand')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# XGBoost
axes[1, 0].plot(y_test, label='Actual', alpha=0.7, linewidth=2)
axes[1, 0].plot(y_pred_xgb, label='Predicted', alpha=0.7, linewidth=2)
axes[1, 0].set_title(f'XGBoost (R²={r2_xgb:.4f})')
axes[1, 0].set_xlabel('Sample')
axes[1, 0].set_ylabel('Demand')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# LSTM (note: different test set size)
axes[1, 1].plot(y_test_lstm, label='Actual', alpha=0.7, linewidth=2)
axes[1, 1].plot(y_pred_lstm, label='Predicted', alpha=0.7, linewidth=2)
axes[1, 1].set_title(f'LSTM (R²={r2_lstm:.4f})')
axes[1, 1].set_xlabel('Sample')
axes[1, 1].set_ylabel('Demand')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 12: Prediction Visualization

In [ ]:
# Aggregate results (note LSTM uses different test set size due to sequences)
comparison_data = {
    'model': ['Linear Regression', 'Random Forest', 'XGBoost', 'LSTM'],
    'rmse': [rmse_lr, rmse_rf, rmse_xgb, rmse_lstm],
    'mae': [mae_lr, mae_rf, mae_xgb, mae_lstm],
    'r2': [r2_lr, r2_rf, r2_xgb, r2_lstm]
}

comparison_df = pd.DataFrame(comparison_data).sort_values('r2', ascending=False).reset_index(drop=True)

print("Model Comparison:")
print(comparison_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

comparison_df.plot(x='model', y='rmse', kind='bar', ax=axes[0], legend=False, color='steelblue')
axes[0].set_title('RMSE Comparison (Lower is Better)')
axes[0].set_ylabel('RMSE')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)

comparison_df.plot(x='model', y='mae', kind='bar', ax=axes[1], legend=False, color='coral')
axes[1].set_title('MAE Comparison (Lower is Better)')
axes[1].set_ylabel('MAE')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

comparison_df.plot(x='model', y='r2', kind='bar', ax=axes[2], legend=False, color='green')
axes[2].set_title('R² Score Comparison (Higher is Better)')
axes[2].set_ylabel('R² Score')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

## Step 11: Model Comparison

In [ ]:
# Reshape for LSTM (samples, lookback, 1)
X_train_lstm_3d = X_train_lstm.reshape((X_train_lstm.shape[0], X_train_lstm.shape[1], 1))
X_val_lstm_3d = X_val_lstm.reshape((X_val_lstm.shape[0], X_val_lstm.shape[1], 1))
X_test_lstm_3d = X_test_lstm.reshape((X_test_lstm.shape[0], X_test_lstm.shape[1], 1))

# Build LSTM
lstm_model = Sequential([
    LSTM(128, activation='relu', return_sequences=True, input_shape=(X_train_lstm.shape[1], 1)),
    Dropout(0.2),
    LSTM(64, activation='relu', return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.1),
    Dense(1)
])

lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

# Train with early stopping
callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = lstm_model.fit(
    X_train_lstm_3d, y_train_lstm,
    validation_data=(X_val_lstm_3d, y_val_lstm),
    epochs=50,
    batch_size=32,
    callbacks=[callback],
    verbose=0
)

# Evaluate on test set
y_pred_lstm = lstm_model.predict(X_test_lstm_3d, verbose=0).flatten()

rmse_lstm = np.sqrt(mean_squared_error(y_test_lstm, y_pred_lstm))
mae_lstm = mean_absolute_error(y_test_lstm, y_pred_lstm)
r2_lstm = r2_score(y_test_lstm, y_pred_lstm)

print(f"LSTM Results:")
print(f"  RMSE: {rmse_lstm:.4f}")
print(f"  MAE: {mae_lstm:.4f}")
print(f"  R² Score: {r2_lstm:.4f}")
print(f"  Epochs trained: {len(history.epoch)}")

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('LSTM Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_title('LSTM Training MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: LSTM Model with Sliding Windows

In [ ]:
xgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBRegressor(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ))
])

xgb_pipeline.fit(X_train, y_train)
y_pred_xgb = xgb_pipeline.predict(X_test)

rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"XGBoost Results:")
print(f"  RMSE: {rmse_xgb:.4f}")
print(f"  MAE: {mae_xgb:.4f}")
print(f"  R² Score: {r2_xgb:.4f}")

# Cross-validation
cv_scores = cross_val_score(xgb_pipeline, X_train, y_train, cv=5, scoring='r2')
print(f"  CV R² Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## Step 9: XGBoost Regressor

In [ ]:
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(
        n_estimators=200,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest Results:")
print(f"  RMSE: {rmse_rf:.4f}")
print(f"  MAE: {mae_rf:.4f}")
print(f"  R² Score: {r2_rf:.4f}")

# Cross-validation
cv_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=5, scoring='r2')
print(f"  CV R² Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## Step 8: Random Forest Regressor

In [ ]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Linear Regression Results:")
print(f"  RMSE: {rmse_lr:.4f}")
print(f"  MAE: {mae_lr:.4f}")
print(f"  R² Score: {r2_lr:.4f}")

# Cross-validation score
cv_scores = cross_val_score(lr_pipeline, X_train, y_train, cv=5, scoring='r2')
print(f"  CV R² Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## Step 7: Linear Regression Model

In [ ]:
class SlidingWindowGenerator:
    """Create sliding window sequences for LSTM."""
    
    def __init__(self, lookback=30, lookahead=1):
        self.lookback = lookback
        self.lookahead = lookahead
    
    def create_sequences(self, data, targets=None):
        """Create sequences."""
        X_seqs = []
        y_seqs = []
        
        for i in range(len(data) - self.lookback - self.lookahead + 1):
            X_seqs.append(data[i:i+self.lookback])
            if targets is not None:
                y_seqs.append(targets[i + self.lookback + self.lookahead - 1])
        
        return np.array(X_seqs), np.array(y_seqs) if targets is not None else None

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Create sliding windows
window_gen = SlidingWindowGenerator(lookback=30, lookahead=1)

X_train_lstm, y_train_lstm = window_gen.create_sequences(X_train_scaled, y_train)
X_val_lstm, y_val_lstm = window_gen.create_sequences(X_val_scaled, y_val)
X_test_lstm, y_test_lstm = window_gen.create_sequences(X_test_scaled, y_test)

print(f"LSTM sequences created:")
print(f"  X_train_lstm shape: {X_train_lstm.shape}")
print(f"  y_train_lstm shape: {y_train_lstm.shape}")
print(f"  X_test_lstm shape: {X_test_lstm.shape}")
print(f"  y_test_lstm shape: {y_test_lstm.shape}")

## Step 6: Sliding Window Generator for LSTM

In [ ]:
# Split time-series preserving temporal order
n = len(df_engineered)
train_end = int(n * 0.7)
val_end = train_end + int(n * 0.15)

df_train = df_engineered.iloc[:train_end].reset_index(drop=True)
df_val = df_engineered.iloc[train_end:val_end].reset_index(drop=True)
df_test = df_engineered.iloc[val_end:].reset_index(drop=True)

print(f"Train: {len(df_train)} samples ({len(df_train)/n*100:.1f}%)")
print(f"Val: {len(df_val)} samples ({len(df_val)/n*100:.1f}%)")
print(f"Test: {len(df_test)} samples ({len(df_test)/n*100:.1f}%)")

# Prepare features
feature_cols = [c for c in df_engineered.columns if c not in ['date', 'store', 'item', 'demand']]

X_train = df_train[feature_cols].values
y_train = df_train['demand'].values

X_val = df_val[feature_cols].values
y_val = df_val['demand'].values

X_test = df_test[feature_cols].values
y_test = df_test['demand'].values

print(f"\nFeature columns: {len(feature_cols)}")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

## Step 5: Time-Series Train/Test Split

In [ ]:
class TimeSeriesFeatureEngineer:
    """Engineer time-series features with fit-transform pattern."""
    
    def __init__(self, lag_windows=[7, 14, 30], rolling_windows=[7, 14, 30]):
        self.lag_windows = lag_windows
        self.rolling_windows = rolling_windows
        self.demand_mean = 1.0
        self.demand_std = 1.0
        self._is_fitted = False
    
    def fit(self, df):
        """Fit on training data."""
        self.demand_mean = df['demand'].mean()
        self.demand_std = max(df['demand'].std(), 1.0)
        self._is_fitted = True
        return self
    
    def create_lag_features(self, series):
        """Create lagged demand features."""
        df = pd.DataFrame({'demand': series})
        for lag in self.lag_windows:
            df[f'demand_lag_{lag}'] = series.shift(lag)
        return df
    
    def create_rolling_features(self, series):
        """Create rolling statistics."""
        df = pd.DataFrame()
        for window in self.rolling_windows:
            df[f'demand_rolling_mean_{window}'] = series.rolling(window=window, min_periods=1).mean()
            df[f'demand_rolling_std_{window}'] = series.rolling(window=window, min_periods=1).std().fillna(0)
        return df
    
    def create_temporal_features(self, dates):
        """Create temporal features."""
        df = pd.DataFrame()
        df['day_of_week'] = dates.dt.dayofweek
        df['month'] = dates.dt.month
        df['quarter'] = dates.dt.quarter
        df['day_of_year'] = dates.dt.dayofyear
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        return df
    
    def transform(self, df):
        """Transform with engineered features."""
        if not self._is_fitted:
            raise ValueError("Must fit first!")
        
        df = df.copy()
        lag_df = self.create_lag_features(df['demand'])
        rolling_df = self.create_rolling_features(df['demand'])
        temporal_df = self.create_temporal_features(df['date'])
        
        df_eng = pd.concat([df, lag_df, rolling_df, temporal_df], axis=1)
        df_eng = df_eng.dropna()
        
        return df_eng
    
    def fit_transform(self, df):
        """Fit and transform."""
        return self.fit(df).transform(df)

# Example usage
fe = TimeSeriesFeatureEngineer()
df_engineered = fe.fit_transform(df_clean)

print(f"Original features: {df_clean.shape[1]}")
print(f"Engineered features: {df_engineered.shape[1]}")
print(f"Feature columns: {[c for c in df_engineered.columns if c not in ['date', 'store', 'item', 'demand', 'day_of_week', 'month']][:10]}")
print(f"\nFirst few rows:\n{df_engineered.head()}")

## Step 4: Feature Engineering for Time-Series

In [ ]:
# Distribution and seasonality patterns
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Demand distribution
axes[0, 0].hist(df_clean['demand'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_title('Demand Distribution')
axes[0, 0].set_xlabel('Demand')
axes[0, 0].set_ylabel('Frequency')

# Demand by day of week (seasonality)
df_clean['day_of_week'] = df_clean['date'].dt.dayofweek
df_clean.groupby('day_of_week')['demand'].mean().plot(kind='bar', ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Average Demand by Day of Week')
axes[0, 1].set_xlabel('Day of Week (0=Monday, 6=Sunday)')
axes[0, 1].set_ylabel('Average Demand')

# Demand by month (seasonality)
df_clean['month'] = df_clean['date'].dt.month
df_clean.groupby('month')['demand'].mean().plot(kind='line', ax=axes[1, 0], marker='o', color='steelblue')
axes[1, 0].set_title('Average Demand by Month')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Average Demand')
axes[1, 0].grid(True, alpha=0.3)

# Demand trends over time (aggregated)
daily_demand = df_clean.groupby('date')['demand'].sum()
axes[1, 1].plot(daily_demand.index, daily_demand.values, color='steelblue', linewidth=1)
axes[1, 1].set_title('Total Daily Demand Over Time')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Total Demand')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Time series plot for each store
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
stores_to_plot = df_clean['store'].unique()[:6]

for idx, store in enumerate(stores_to_plot):
    ax = axes[idx // 3, idx % 3]
    store_data = df_clean[df_clean['store'] == store].sort_values('date')
    
    for item in df_clean['item'].unique()[:2]:  # Plot first 2 items per store
        item_data = store_data[store_data['item'] == item]
        ax.plot(item_data['date'], item_data['demand'], label=f'Item {item}', alpha=0.7)
    
    ax.set_title(f'Store {store} Demand Over Time')
    ax.set_xlabel('Date')
    ax.set_ylabel('Demand')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Demand distribution by store and item
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_clean.groupby('store')['demand'].mean().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Average Demand by Store')
axes[0].set_xlabel('Store')
axes[0].set_ylabel('Average Demand')
axes[0].grid(True, alpha=0.3)

df_clean.groupby('item')['demand'].mean().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Average Demand by Item')
axes[1].set_xlabel('Item')
axes[1].set_ylabel('Average Demand')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 3: Time-Series Exploratory Analysis

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean dataset: remove duplicates, handle missing values."""
    df = df.copy()
    initial_len = len(df)
    
    df = df.drop_duplicates(subset=['date', 'store', 'item'], keep='first')
    df = df.dropna()
    
    print(f"Removed {initial_len - len(df)} rows (duplicates/missing)")
    return df

df_clean = clean_data(df)
print(f"After cleaning - shape: {df_clean.shape}")
print(f"Demand statistics: min={df_clean['demand'].min():.2f}, max={df_clean['demand'].max():.2f}, mean={df_clean['demand'].mean():.2f}")

## Step 2: Data Cleaning & Preprocessing

In [ ]:
# Generate synthetic Store Item Demand dataset
np.random.seed(42)

# Create time series data
dates = pd.date_range('2020-01-01', periods=1000, freq='D')
stores = [1, 2, 3, 4, 5]
items = [1, 2, 3, 4, 5]

data = []
for date in dates:
    for store in stores:
        for item in items:
            # Generate realistic demand with trend and seasonality
            day = (date - dates[0]).days
            trend = day * 0.1
            seasonality = 20 * np.sin(2 * np.pi * day / 365)
            noise = np.random.normal(0, 5)
            demand = max(10, 50 + trend + seasonality + noise + store * 5 + item * 3)
            data.append({
                'date': date,
                'store': store,
                'item': item,
                'demand': demand
            })

df = pd.DataFrame(data)

print(f"Dataset shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nFirst few rows:\n{df.head()}")
print(f"\nBasic statistics:\n{df.describe()}")
print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

## Step 1: Load Dataset

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import mlflow
import mlflow.sklearn
import mlflow.keras
import joblib

print("Libraries imported successfully")

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Environment configured")

# Demand Forecasting - Store Item Demand EDA & Model Training

End-to-end time-series exploratory data analysis and model training for Store Item Demand forecasting.

**Dataset**: Store Item Demand - Multi-store, multi-item daily sales forecasting  
**Models**: Linear Regression, Random Forest Regressor, XGBoost Regressor, LSTM  
**Goal**: Build, train, evaluate, and persist regression models with time-series features and MLflow integration